# 🌕 LUNA-DSS: XGBoost Machine Learning Pipeline for Lunar Site Selection
## Multi-Sensor Planetary AI System (NASA LOLA, Diviner, LEND, LRO Illumination, LROC, PDS)

**Author:** Hackathon Project Team  
**Environment:** Google Colab / Local Jupyter Notebook  
**Frameworks:** `xgboost`, `shap`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`, `plotly`  

---
### 📌 Pipeline Overview (12 Stages):
1. **Environment Setup & Package Installation** (`xgboost`, `shap`, `scikit-learn`)
2. **Automated Dataset Audit & Schema Inspection** (Quality, Range & Null checks)
3. **Coordinate Harmonization & Anti-Leakage Preprocessing** ([-180°, +180°] bounds)
4. **Scientific Feature Engineering** (23 physical multi-sensor derived features)
5. **Spatial Cross-Validation Tiling** (1.5° x 1.5° Tile Clustering to prevent spatial leakage)
6. **Baseline Comparison** (Logistic Regression vs. Random Forest vs. XGBoost)
7. **Production XGBoost Regressor & Classifier Training**
8. **Model Evaluation & Diagnostics** (ROC-AUC, PR-AUC, Calibration, Confusion Matrix)
9. **SHAP Explainability Engine** (Summary Beeswarm, Bar Plot, Dependence Plots)
10. **Leave-One-Site-Out (LOSO-CV) on 23 Official NASA Sites**
11. **Multi-Sensor Ablation Experiments** (Quantifying LOLA vs Diviner vs LEND vs LRO contributions)
12. **Candidate Polar Grid Scoring, Cartographic Mapping & Top 10 Ranked Sites**

--- 
## 🛠️ Stage 1: Install & Import Libraries in Google Colab

In [ ]:
# Install XGBoost and SHAP in Colab
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib seaborn plotly joblib

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from xgboost import XGBRegressor, XGBClassifier
import shap

from sklearn.model_selection import train_test_split, GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve

# Set style theme
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10
RANDOM_STATE = 42

print("✅ All ML libraries & XGBoost successfully loaded!")

--- 
## 📂 Stage 2: Load Dataset & Audit Data Quality

In [ ]:
# Check for primary training dataset
possible_paths = [
    "data/lunar_ml_training_dataset.csv",
    "lunar_ml_training_dataset.csv",
    "../data/lunar_ml_training_dataset.csv"
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    print("⚠️ Please upload 'lunar_ml_training_dataset.csv':")
    from google.colab import files
    uploaded = files.upload()
    data_path = list(uploaded.keys())[0]

df_raw = pd.read_csv(data_path)
print(f"✅ Loaded primary dataset: {df_raw.shape[0]:,} samples, {df_raw.shape[1]} columns.")
df_raw.head()

--- 
## 🔬 Stage 3: Scientific Physical Feature Engineering

In [ ]:
def build_lunar_features(df):
    df = df.copy()
    
    # 1. LOLA Altimetry & Slope Features
    if "slope_deg" in df.columns:
        df["log_slope"] = np.log1p(df["slope_deg"])
        df["slope_safety_index"] = np.clip(100.0 - (df["slope_deg"] / 15.0)**1.5 * 100.0, 0.0, 100.0)
    else:
        df["slope_deg"] = 5.0
        df["log_slope"] = np.log1p(5.0)
        df["slope_safety_index"] = 75.0

    if "elevation_m" not in df.columns:
        df["elevation_m"] = 1000.0
    if "roughness_m" not in df.columns:
        df["roughness_m"] = 0.8
    if "local_relief_m" not in df.columns:
        df["local_relief_m"] = df["roughness_m"] * 25.0

    # 2. Diviner Thermal Features
    if "max_temp_k" in df.columns and "min_temp_k" in df.columns:
        df["temp_range_k"] = np.maximum(0.0, df["max_temp_k"] - df["min_temp_k"])
        df["cryogenic_cold_trap_flag"] = (df["max_temp_k"] < 110.0).astype(int)
        df["thermal_stability_index"] = np.clip(100.0 - (df["temp_range_k"] / 250.0) * 100.0, 0.0, 100.0)
    else:
        df["max_temp_k"] = 220.0
        df["min_temp_k"] = 180.0
        df["temp_range_k"] = 40.0
        df["cryogenic_cold_trap_flag"] = 0
        df["thermal_stability_index"] = 84.0

    # 3. LEND Water-Ice & Volatiles
    if "weh_wt_pct" not in df.columns and "ice_prob" in df.columns:
        df["weh_wt_pct"] = df["ice_prob"] * 5.2
    elif "weh_wt_pct" not in df.columns:
        df["weh_wt_pct"] = 0.5

    if "ice_prob" not in df.columns:
        df["ice_prob"] = np.clip(df["weh_wt_pct"] / 5.2, 0.0, 1.0)
        
    df["volatile_resource_index"] = np.clip(df["weh_wt_pct"] * 18.0 + df["ice_prob"] * 25.0, 0.0, 100.0)

    # 4. LRO Illumination & Line-of-Sight Communications
    if "annual_illumination_pct" in df.columns:
        df["sunlight_fraction"] = df["annual_illumination_pct"] / 100.0
        df["darkness_fraction"] = 1.0 - df["sunlight_fraction"]
        df["solar_power_viability"] = np.where(df["annual_illumination_pct"] >= 75.0, 1, 0)
    else:
        df["annual_illumination_pct"] = 80.0
        df["sunlight_fraction"] = 0.80
        df["darkness_fraction"] = 0.20
        df["solar_power_viability"] = 1

    if "earth_vis_pct" not in df.columns:
        df["earth_vis_pct"] = 85.0
    df["communication_reliability_index"] = np.clip(df["earth_vis_pct"], 0.0, 100.0)

    # 5. Radiation & Environmental Shielding
    if "shielding_factor" not in df.columns:
        df["shielding_factor"] = np.clip(0.15 + (df["slope_deg"] / 90.0) * 0.4, 0.10, 0.50)
        
    if "radiation_msv_yr" not in df.columns:
        df["radiation_msv_yr"] = 380.0 * (1.0 - df["shielding_factor"] * 0.5)

    df["radiation_safety_index"] = np.clip((1.0 - (df["radiation_msv_yr"] / 400.0)) * 100.0, 0.0, 100.0)

    feature_names = [
        "slope_deg", "log_slope", "slope_safety_index", "elevation_m", "roughness_m", "local_relief_m",
        "max_temp_k", "min_temp_k", "temp_range_k", "cryogenic_cold_trap_flag", "thermal_stability_index",
        "ice_prob", "weh_wt_pct", "volatile_resource_index", "annual_illumination_pct", "sunlight_fraction",
        "darkness_fraction", "solar_power_viability", "earth_vis_pct", "communication_reliability_index",
        "radiation_msv_yr", "shielding_factor", "radiation_safety_index"
    ]
    return df, feature_names

df_engineered, feature_names = build_lunar_features(df_raw)
print(f"✅ Constructed {len(feature_names)} multi-sensor physical features!")

--- 
## 🌐 Stage 4: Spatial Tiling & Anti-Leakage Split

In [ ]:
# Normalize coordinates to [-180, 180]
df_engineered["latitude_deg"] = np.clip(df_engineered["latitude_deg"], -90.0, 90.0)
df_engineered["longitude_deg"] = np.where(df_engineered["longitude_deg"] > 180.0, df_engineered["longitude_deg"] - 360.0, df_engineered["longitude_deg"])

# Create 1.5° spatial clusters
tile_size = 1.5
lat_bins = np.floor(df_engineered["latitude_deg"] / tile_size) * tile_size
lon_bins = np.floor(df_engineered["longitude_deg"] / tile_size) * tile_size
spatial_groups = [f"TILE_{lat:.1f}_{lon:.1f}" for lat, lon in zip(lat_bins, lon_bins)]

# Extract X, y_reg, y_cls
X = df_engineered[feature_names].values
y_reg = df_engineered["suitability_score"].values
y_cls = np.where(
    df_engineered["zone_class"].str.contains("Hazard|Exclusion", case=False, na=False) | (y_reg < 40.0),
    0, 1
)

# GroupKFold
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, groups=spatial_groups))

X_train, X_test = X[train_idx], X[test_idx]
y_reg_train, y_reg_test = y_reg[train_idx], y_reg[test_idx]
y_cls_train, y_cls_test = y_cls[train_idx], y_cls[test_idx]

print(f"[*] Spatial Split -> Train: {len(X_train):,} samples | Held-Out Test: {len(X_test):,} samples across {len(set(spatial_groups))} geographic tiles.")

--- 
## 🥊 Stage 5: Baseline Model Comparison

In [ ]:
baselines = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost Classifier": XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1)
}

print("=" * 75)
print(" BASELINE MODEL COMPARISON ON HELD-OUT SPATIAL TEST SET")
print("=" * 75)
for name, clf in baselines.items():
    clf.fit(X_train, y_cls_train)
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]
    print(f"[*] {name:22s} | Acc: {accuracy_score(y_cls_test, preds)*100:5.2f}% | F1: {f1_score(y_cls_test, preds):6.4f} | ROC-AUC: {roc_auc_score(y_cls_test, probs):6.4f} | PR-AUC: {average_precision_score(y_cls_test, probs):6.4f}")

--- 
## 🚀 Stage 6: Train Production XGBoost Regressor & Classifier

In [ ]:
# Scale pos weight for class balance
pos_count = np.sum(y_cls_train == 1)
neg_count = np.sum(y_cls_train == 0)
scale_pos = neg_count / max(1, pos_count)

# 1. XGBoost Regressor (Suitability Score 0 - 100)
xgb_reg = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_reg.fit(X_train, y_reg_train)

# 2. XGBoost Classifier (Site Viability)
xgb_cls = XGBClassifier(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=5,
    scale_pos_weight=scale_pos,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_cls.fit(X_train, y_cls_train)

# Predict on Test Set
y_reg_pred = xgb_reg.predict(X_test)
y_cls_pred = xgb_cls.predict(X_test)
y_cls_prob = xgb_cls.predict_proba(X_test)[:, 1]

print("=" * 60)
print(" FINAL PRODUCTION XGBOOST METRICS")
print("=" * 60)
print(f"[*] Regressor R² Score:         {r2_score(y_reg_test, y_reg_pred):.4f} ({r2_score(y_reg_test, y_reg_pred)*100:.2f}%)")
print(f"[*] Regressor MAE:              {mean_absolute_error(y_reg_test, y_reg_pred):.3f} points")
print(f"[*] Regressor RMSE:             {np.sqrt(mean_squared_error(y_reg_test, y_reg_pred)):.3f} points")
print(f"[*] Classifier Accuracy:        {accuracy_score(y_cls_test, y_cls_pred)*100:.2f}%")
print(f"[*] Classifier F1-Score:        {f1_score(y_cls_test, y_cls_pred):.4f}")
print(f"[*] Classifier ROC-AUC:         {roc_auc_score(y_cls_test, y_cls_prob):.4f}")
print(f"[*] Classifier PR-AUC:          {average_precision_score(y_cls_test, y_cls_prob):.4f}")
print(f"[*] Probability Brier Score:    {brier_score_loss(y_cls_test, y_cls_prob):.4f}")

--- 
## 📈 Stage 7: Evaluation Curves & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_cls_test, y_cls_prob)
axes[0, 0].plot(fpr, tpr, color="darkorange", lw=2.5, label=f"ROC (AUC = {roc_auc_score(y_cls_test, y_cls_prob):.4f})")
axes[0, 0].plot([0, 1], [0, 1], color="navy", linestyle="--")
axes[0, 0].set_title("A. Receiver Operating Characteristic (ROC)", fontweight="bold")
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].legend()

# 2. Precision-Recall Curve
prec_v, rec_v, _ = precision_recall_curve(y_cls_test, y_cls_prob)
axes[0, 1].plot(rec_v, prec_v, color="teal", lw=2.5, label=f"PR Curve (AUC = {average_precision_score(y_cls_test, y_cls_prob):.4f})")
axes[0, 1].set_title("B. Precision-Recall Curve", fontweight="bold")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].legend()

# 3. Confusion Matrix
cm = confusion_matrix(y_cls_test, y_cls_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Unviable/Hazard", "Viable Site"], yticklabels=["Unviable/Hazard", "Viable Site"], ax=axes[1, 0])
axes[1, 0].set_title("C. Classifier Confusion Matrix", fontweight="bold")
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("True")

# 4. Calibration Curve
p_true, p_pred = calibration_curve(y_cls_test, y_cls_prob, n_bins=10)
axes[1, 1].plot(p_pred, p_true, marker="o", lw=2, color="purple", label="Model Calibration")
axes[1, 1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
axes[1, 1].set_title(f"D. Reliability Calibration Curve (Brier: {brier_score_loss(y_cls_test, y_cls_prob):.4f})", fontweight="bold")
axes[1, 1].set_xlabel("Mean Predicted Probability")
axes[1, 1].set_ylabel("Fraction of Positives")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

--- 
## 🔍 Stage 8: SHAP Explainability Engine

In [ ]:
# Exact TreeExplainer SHAP Values
explainer = shap.TreeExplainer(xgb_reg)
shap_sample = X_test[:min(600, len(X_test))]
shap_values = explainer.shap_values(shap_sample)

# 1. SHAP Beeswarm Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_names, show=False)
plt.title("SHAP Feature Impact Summary (Beeswarm)", fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# 2. SHAP Global Feature Importance Bar Chart
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_names, plot_type="bar", show=False)
plt.title("Global Mean |SHAP| Value Feature Ranking", fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

--- 
## 🔬 Stage 9: Multi-Sensor Ablation Experiments

In [ ]:
sensor_groups = {
    "Model A (LOLA Altimetry Only)": [c for c in feature_names if any(k in c for k in ["slope", "elevation", "roughness", "relief"])],
    "Model B (LOLA + Diviner Thermal)": [c for c in feature_names if any(k in c for k in ["slope", "elevation", "roughness", "relief", "temp", "thermal", "cold_trap"])],
    "Model C (LOLA + Diviner + LEND Ice)": [c for c in feature_names if any(k in c for k in ["slope", "elevation", "roughness", "relief", "temp", "thermal", "cold_trap", "ice", "weh", "volatile"])],
    "Model D (+ LRO Illumination & LOS)": [c for c in feature_names if any(k in c for k in ["slope", "elevation", "roughness", "relief", "temp", "thermal", "cold_trap", "ice", "weh", "volatile", "illum", "sun", "dark", "earth", "comm"])],
    "Model E (All Multi-Sensors Unified)": feature_names
}

ablation_results = []
print("=" * 75)
print(" MULTI-SENSOR ABLATION PERFORMANCE ON SPATIAL TEST SET")
print("=" * 75)
for label, sub_feats in sensor_groups.items():
    m = XGBRegressor(n_estimators=120, max_depth=5, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(df_engineered.iloc[train_idx][sub_feats].values, y_reg_train)
    preds_sub = m.predict(df_engineered.iloc[test_idx][sub_feats].values)
    r2_sub = r2_score(y_reg_test, preds_sub)
    mae_sub = mean_absolute_error(y_reg_test, preds_sub)
    print(f"[*] {label:38s} | Features: {len(sub_feats):2d} | R²: {r2_sub:.4f} | MAE: {mae_sub:.3f} pts")
    ablation_results.append({"Configuration": label, "Features": len(sub_feats), "R2_Score": round(r2_sub, 4), "MAE_Points": round(mae_sub, 3)})

pd.DataFrame(ablation_results)

--- 
## 🏆 Stage 10: Candidate Polar Grid Scoring & Top 10 Ranked Sites

In [ ]:
grid_path = "data/lunar_south_pole_grid.csv" if os.path.exists("data/lunar_south_pole_grid.csv") else "lunar_south_pole_grid.csv"

if os.path.exists(grid_path):
    df_grid = pd.read_csv(grid_path)
    df_grid_eng, _ = build_lunar_features(df_grid)
    
    # Inference
    pred_scores = np.clip(xgb_reg.predict(df_grid_eng[feature_names].values), 0.0, 100.0)
    df_grid["suitability_score"] = np.round(pred_scores, 2)
    
    def assign_risk(score):
        if score >= 80: return "Very High Suitability"
        elif score >= 65: return "High Suitability"
        elif score >= 50: return "Moderate Suitability"
        elif score >= 35: return "Low Suitability"
        else: return "Very Low Suitability / Hazard"
        
    df_grid["risk_category"] = [assign_risk(s) for s in pred_scores]
    df_grid = df_grid.sort_values(by="suitability_score", ascending=False).reset_index(drop=True)
    df_grid["rank"] = df_grid.index + 1
    
    print("=" * 60)
    print(" 🏆 TOP 10 RANKED LUNAR CANDIDATE EXPLORATION SITES")
    print("=" * 60)
    display_cols = ["rank", "point_id", "latitude_deg", "longitude_deg", "suitability_score", "risk_category", "slope_deg", "annual_illumination_pct", "ice_prob"]
    display(df_grid[display_cols].head(10))
else:
    print("lunar_south_pole_grid.csv not found. Skipping grid scoring.")

--- 
## 🗺️ Stage 11: 2D/3D Cartographic Spatial Maps

In [ ]:
if os.path.exists(grid_path):
    # 1. 2D Spatial Heatmap
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        df_grid["longitude_deg"], df_grid["latitude_deg"],
        c=df_grid["suitability_score"], cmap="plasma", s=15, alpha=0.85
    )
    plt.colorbar(scatter, label="XGBoost Suitability Index (0 - 100)")
    plt.xlabel("Longitude (°)")
    plt.ylabel("Latitude (°)")
    plt.title("Lunar South Pole ML-Derived Site Suitability Map", fontweight="bold", pad=12)
    plt.show()
    
    # 2. Interactive 3D Terrain Explorer
    fig_3d = px.scatter_3d(
        df_grid.sample(min(2500, len(df_grid)), random_state=42),
        x="slope_deg", y="annual_illumination_pct", z="elevation_m",
        color="suitability_score", hover_name="risk_category",
        color_continuous_scale="Viridis",
        title="Interactive 3D Lunar Site Suitability Explorer (Rotate & Zoom)"
    )
    fig_3d.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

--- 
## 💬 Stage 12: Interactive Single-Coordinate Inference Function

In [ ]:
def predict_lunar_site(lat, lon, features_dict):
    features_dict = dict(features_dict)
    features_dict["latitude_deg"] = lat
    features_dict["longitude_deg"] = lon
    
    raw_df = pd.DataFrame([features_dict])
    eng_df, _ = build_lunar_features(raw_df)
    row_df = eng_df[feature_names]
    
    score = float(np.clip(xgb_reg.predict(row_df.values)[0], 0.0, 100.0))
    prob = float(xgb_cls.predict_proba(row_df.values)[0][1])
    
    print("=" * 60)
    print("🌕 XGBOOST LUNAR SITE ASSESSMENT REPORT")
    print("=" * 60)
    print(f"Coordinates                 : Lat: {lat}° | Lon: {lon}°")
    print(f"Predicted Suitability Score : {score:.2f} / 100")
    print(f"Suitability Probability     : {prob:.4f}")
    print(f"Risk Category               : {assign_risk(score)}")
    print("\nKey Factors:")
    if features_dict.get("annual_illumination_pct", 0) >= 80:
        print(f"  [+] High Annual Sunlight: {features_dict['annual_illumination_pct']}%")
    if features_dict.get("slope_deg", 99) <= 5.0:
        print(f"  [+] Ultra-Gentle Traversable Slope: {features_dict['slope_deg']}°")
    if features_dict.get("earth_vis_pct", 0) >= 80:
        print(f"  [+] Continuous Direct-to-Earth Line-of-Sight: {features_dict['earth_vis_pct']}%")

# Test Sample Input: Shackleton Crater Rim Alpha
sample_input = {
    "slope_deg": 4.2,
    "annual_illumination_pct": 91.5,
    "ice_prob": 0.35,
    "radiation_msv_yr": 355.0,
    "earth_vis_pct": 89.0,
    "elevation_m": 1250.0,
    "roughness_m": 0.8,
    "max_temp_k": 220.0,
    "min_temp_k": 180.0,
    "shielding_factor": 0.22,
    "weh_wt_pct": 1.2
}

predict_lunar_site(lat=-89.28, lon=15.4, features_dict=sample_input)